# 📐 Evaluation als Spezifikation

> *"In klassischer Software definieren Tests die Korrektheit. In KI-Software definieren **Metriken** die Korrektheit."*

Deine Metrik-Funktion IST deine formale Spezifikation. Sie sagt dem System, was "richtig" bedeutet. Ohne Metrik bist du blind.

Der Weg ist klar: **Bau dir eine Metrik → teste sie mit Fake-Daten → lass das echte Modell laufen → sieh die Scores.** Von "Ich weiss nicht ob's gut ist" zu "Ich kann exakt messen wie gut."

| Klassische Software | KI-Software |
|---|---|
| Unit Test | Metrik-Funktion |
| `assert x == y` | `metric(expected, predicted) → score` |
| Pass/Fail (binär) | 0.0 bis 1.0 (kontinuierlich) |
| Testet Code | Testet Modell-Output |

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy
import ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.calculations import (METRIC_REGISTRY, code_execution_proxy, analogy_match,
                                      fact_verdict_accuracy, numeric_match)
from dspy_tasks.actions import run_baseline, _evaluate_examples, _mean
from dspy_tasks.visualize import *
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())
display(model_dd)

In [ ]:
from dspy_tasks.visualize import diagram_compare

diagram_compare(
    {"title": "Klassische Software", "items": ["Unit Test", "assert x == y", "Pass/Fail"], "icon": "🔧", "color": "#8a8886"},
    {"title": "KI-Software", "items": ["Metrik-Funktion", "metric(gold, pred) → score", "0.0 bis 1.0"], "icon": "🧠", "color": "#0078d4"},
    title="Tests vs. Metriken"
)

## Metriken werden immer raffinierter

Von simplem Exact-Match über Token-F1 bis hin zu gewichteten Composite-Scores — je besser deine Metrik, desto präziser kannst du optimieren und vergleichen.

In [ ]:
# Level 1: Binary exact match (simplest)
from IPython.display import display, HTML
from dspy_tasks.calculations import token_f1

f1_val = token_f1(['apple','banana'], ['apple','cherry'])

display(HTML(
    '<div style="margin:8px 0">'
    '<div style="padding:12px; background:#f3f2f1; border-radius:8px; margin-bottom:8px; border-left:4px solid #107c10">'
    '<b style="font-size:1.05em">Level 1: Exact Match</b>'
    '<div style="margin-top:6px; font-family:monospace; font-size:0.95em">'
    '  'positive' == 'positive' → <span style="color:#107c10; font-weight:bold">1.0</span><br>'
    '  'positive' == 'POSITIVE' → <span style="color:#107c10; font-weight:bold">1.0</span>  <span style="color:#605e5c">(normalized)</span><br>'
    '  'positive' == 'negative' → <span style="color:#d13438; font-weight:bold">0.0</span></div></div>'
    '<div style="padding:12px; background:#f3f2f1; border-radius:8px; margin-bottom:8px; border-left:4px solid #0078d4">'
    '<b style="font-size:1.05em">Level 2: Token F1</b>'
    f'<div style="margin-top:6px; font-family:monospace; font-size:0.95em">'
    f'  gold=['apple','banana'], pred=['apple','cherry'] → <span style="color:#0078d4; font-weight:bold">{f1_val:.3f}</span></div></div>'
    '<div style="padding:12px; background:#f3f2f1; border-radius:8px; border-left:4px solid #8764b8">'
    '<b style="font-size:1.05em">Level 3: Composite (ticket routing)</b>'
    '<div style="margin-top:6px; font-size:0.95em">'
    '  Priority correct (<b>40%</b>) + Category correct (<b>35%</b>) + Team correct (<b>25%</b>)<br>'
    '  = Weighted specification of "what matters most"</div></div></div>'
))

display_insight("Der Kern-Insight",
    "Deine Metrik-Funktion IST deine Produkt-Spezifikation. "
    "Wenn du die Gewichte änderst, änderst du, wofür das System optimiert. "
    "Deshalb ist 'Evaluation die Software der Zukunft'.")

In [ ]:
from dspy_tasks.config import configure_dspy
from IPython.display import display, HTML
# Code Generation task — different metric than simple matching
task = get_task("code_generation")
display(HTML(
    f'<div style="padding:12px; background:#f3f2f1; border-radius:8px; margin:8px 0">'
    f'<div style="font-size:1.1em"><b>Task:</b> {task.name}</div>'
    f'<div style="margin-top:4px"><b>Metric:</b> <code>code_execution_proxy</code> '
    f'<span style="color:#605e5c">(checks structure, keywords, overlap)</span></div>'
    f'<div style="margin-top:4px"><b>Teaching point:</b> {task.teaching_point}</div></div>'
))

btn = run_button("Evaluate Code Generation")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        configure_dspy(model=model_dd.value)
        result = run_baseline("code_generation", model_dd.value, max_eval=8)
        display_score("Code Generation", result.score)
        display_results_table(result.individual_scores)

btn.on_click(on_run)
display(widgets.HBox([model_dd, btn]), out)

In [ ]:
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task(tid) for tid in ["code_generation", "analogy", "fact_verification"]]],
    description="Task:")
compare_btn = run_button("Compare All Models")
compare_out = widgets.Output()

def on_compare(b):
    with compare_out:
        compare_out.clear_output()
        print(f"⏳ Evaluating {task_dd.value} across {len(MODELS)} models...")
        scores = {}
        for m in MODELS:
            result = run_baseline(task_dd.value, m, max_eval=8)
            scores[m] = {"baseline": result.score}
            display_score(m.split("/")[-1], result.score)

        fig = bar_comparison(get_task(task_dd.value).name, scores)
        fig.show()

compare_btn.on_click(on_compare)
display(widgets.HBox([task_dd, compare_btn]), compare_out)

## Verschiedene Metriken, verschiedene Rankings

Das Gleiche Output kann unter verschiedenen Metriken unterschiedlich gut abschneiden. **Deine Metrik zu wählen heisst, deine Werte zu wählen.**

In [ ]:
display_insight("Evaluation = Spezifikation",
    "In klassischer Software schreibst du Tests NACH dem Code. "
    "In KI-Software schreibst du Metriken VOR der Optimierung. "
    "Die Metrik IST die Spezifikation. Der Optimizer findet Code (Prompts), der deine Tests besteht.",
    icon="📐")

## ✏️ Dein Prompt-Tuning Workshop

Jetzt bist DU dran! Unten siehst du eine vorausgefüllte Prompt-Anweisung. Editiere den Text und klick "Auswerten" um zu sehen, wie sich dein Score ändert.

**Tipps für bessere Prompts:**
- Sei spezifischer (z.B. "Antworte mit genau einem Wort: positive, negative, oder neutral")
- Gib Kontext ("Du bist ein erfahrener Produktbewertungs-Analyst")
- Erwähne Sonderfälle ("Achte besonders auf Sarkasmus und Ironie")

Jeder Versuch wird aufgezeichnet — du siehst deinen Fortschritt!

In [ ]:
from dspy_tasks.visualize import prompt_workshop, model_picker
from dspy_tasks.config import get_available_models, get_default_model

# Vorausgefüllter Prompt — editiere ihn und sieh was passiert!
workshop = prompt_workshop(
    task_id="sentiment",
    model_widget=model_dd,
    default_instructions="Classify the sentiment of the product review as positive, negative, or neutral.",
    max_eval=10,
)
display(workshop)

### 💡 Was hast du beobachtet?

- Hat sich dein Score verbessert?
- Welche Formulierungen haben geholfen?
- Wie lange hast du dafür gebraucht?

Das ist **Prompt Engineering** — manuelles Ausprobieren von Formulierungen. Es funktioniert, aber:
- Es ist **zeitaufwändig** (jeder Versuch dauert Sekunden bis Minuten)
- Es ist **fragil** (was bei einem Modell klappt, versagt bei einem anderen)
- Es ist **nicht reproduzierbar** (wie weisst du, dass Version 47 besser war als Version 23?)

> 🤔 **Was wäre, wenn ein Computer das automatisch machen könnte?** Tausende Varianten ausprobieren, jede bewerten, die beste behalten? Das ist Notebook 04!

In [ ]:
from dspy_tasks.benchmarks import load_truthfulqa, contains_match
from dspy_tasks.actions import run_on_examples
import dspy

truthful_examples = load_truthfulqa(8)

# Vorausgefüllter Prompt für TruthfulQA
tqa_prompt = widgets.Textarea(
    value="Answer the question accurately. Be careful about common misconceptions and myths. If the common belief is wrong, give the scientifically correct answer.",
    layout=widgets.Layout(width="100%", height="100px"),
)
tqa_label = widgets.HTML('<div style="font-weight:bold; margin-bottom:4px">✏️ Dein Prompt für Fakten-Fragen:</div>')
tqa_btn = widgets.Button(description="Auswerten!", button_style="primary", icon="play", layout=widgets.Layout(width="200px"))
tqa_out = widgets.Output()

class TQASig(dspy.Signature):
    """Placeholder"""
    question = dspy.InputField(desc="A factual question")
    answer = dspy.OutputField(desc="A truthful answer")

def on_tqa_run(b):
    with tqa_out:
        tqa_out.clear_output()
        result = run_on_examples(
            truthful_examples, tqa_prompt.value, model_dd.value, TQASig, contains_match,
        )
        display_score("TruthfulQA Score", result.score)
        display_results_table(result.individual_scores)

tqa_btn.on_click(on_tqa_run)
display(tqa_label, tqa_prompt, tqa_btn, tqa_out)

## ⏭️ Weiter geht's!

Du hast Metriken. Du kannst messen. Aber was, wenn der **Computer die Prompts SELBST optimieren** könnte? 

Stell dir vor: du schreibst die Spezifikation (was du willst + was "gut" heisst), und das Werkzeug findet den besten Prompt dafür. Genau wie ein Compiler! Das ist Notebook 02.